In [1]:
import numpy as np
import sys
from pympler import asizeof
from timeit import timeit

ModuleNotFoundError: No module named 'pympler'

# general numpy

## bitwise_and and addition

In [2]:
def fun_bit_and():
    x = np.bitwise_and(np.random.randint(low = 0, high = 255, size = 150, dtype = np.int16), 
                       np.random.randint(low = 0, high = 255, size = 150, dtype = np.int16))
    return x

timeit(fun_bit_and, number=1000)

0.06255802884697914

### breakdown

In [3]:
%timeit np.random.randint(low = 0, high = 255, size = 150, dtype = np.int16)

22.2 µs ± 184 ns per loop (mean ± std. dev. of 7 runs, 10000 loops each)


### bitwise_add

In [4]:
x = np.random.randint(low = 0, high = 255, size = 150, dtype = np.int16)
y = np.random.randint(low = 0, high = 255, size = 150, dtype = np.int16)

In [5]:
%timeit np.bitwise_and(x,y)

774 ns ± 3.01 ns per loop (mean ± std. dev. of 7 runs, 1000000 loops each)


### addition

In [6]:
%timeit x+y

585 ns ± 3.89 ns per loop (mean ± std. dev. of 7 runs, 1000000 loops each)


In [7]:
%%timeit -n 1000 array1, array2 = np.random.random((2, 150))
array1 = array1 + array2
# too few rounds, might be inaccurate

837 ns ± 26.7 ns per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [8]:
%%timeit -n 100000 array1, array2 = np.random.randint((2, 150))
array1 = array1 + array2

65.4 ns ± 1.67 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


In [9]:
%%timeit -n 100000 array1, array2 = np.random.randint((2, 150))
array1 += array2

66.1 ns ± 4.08 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


In [10]:
%%timeit -n 1000000 array1, array2 = np.random.randint((2, 150))
array1 += array2

62.9 ns ± 0.708 ns per loop (mean ± std. dev. of 7 runs, 1000000 loops each)


### np.where

In [11]:
%timeit np.where(x==0)

4.72 µs ± 34.4 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


In [12]:
%timeit np.where(x)

2.06 µs ± 29.2 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


## size of different dtype

In [13]:
x = np.array(range(0, 240 * 10**6), dtype = np.int8)
print(sys.getsizeof(x))

240000104


In [14]:
x = np.random.random(size=3000000)

print(sys.getsizeof(x))
print(sys.getsizeof(np.float16(x)))

24000104
6000104


## translate

In [15]:
DNA_dict = {0:'A', 1:'C',2:'G',3:'T'}
my_arr = np.random.randint(4, size=100)

In [16]:
def g():
    return ''.join([DNA_dict[i] for i in my_arr])

In [17]:
%timeit g()

31.9 µs ± 333 ns per loop (mean ± std. dev. of 7 runs, 10000 loops each)


In [18]:
%timeit ''.join('ACGT'[i] for i in my_arr)

15.7 µs ± 157 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


In [19]:
''.join('ACGT'[i] for i in my_arr)

'GGGCCGAGGAAAAATGCGAGTTCAGCAACGCATTTCAGGCATGCAATCGGTCGGCAAGCTGAGGTGGAACTAGGCTCAGTCGATAGCTGTATTCATCTTA'

## create or deepcopy

In [1]:
from copy import deepcopy

In [4]:
%timeit np.full(150, 56)

4.74 µs ± 13.6 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


In [5]:
x = np.full(150, 56)

In [6]:
%timeit deepcopy(x)

3.3 µs ± 78.3 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


# masked array

In [20]:
x = np.arange(10)
m = np.ma.masked_array(x, x>5)

In [21]:
m

masked_array(data=[0, 1, 2, 3, 4, 5, --, --, --, --],
             mask=[False, False, False, False, False, False,  True,  True,
                    True,  True],
       fill_value=999999)

In [22]:
m+1

masked_array(data=[1, 2, 3, 4, 5, 6, --, --, --, --],
             mask=[False, False, False, False, False, False,  True,  True,
                    True,  True],
       fill_value=999999)

## time

In [23]:
%timeit m = np.ma.masked_array(x, x>5)

17.7 µs ± 36.9 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


## size

In [24]:
print(sys.getsizeof(x))
print(sys.getsizeof(m))

184
128


In [25]:
print(asizeof.asizeof(x))
print(asizeof.asizeof(m))

200
328


## operations

In [26]:
np.bitwise_and(m+1, 0x1)

masked_array(data=[1, 0, 1, 0, 1, 0, --, --, --, --],
             mask=[False, False, False, False, False, False,  True,  True,
                    True,  True],
       fill_value=999999)

In [27]:
m & x

masked_array(data=[0, 1, 2, 3, 4, 5, --, --, --, --],
             mask=[False, False, False, False, False, False,  True,  True,
                    True,  True],
       fill_value=999999)

In [28]:
np.sum(np.logical_not(m.mask))

6

In [29]:
np.bitwise_and(m+1, 0x1)

masked_array(data=[1, 0, 1, 0, 1, 0, --, --, --, --],
             mask=[False, False, False, False, False, False,  True,  True,
                    True,  True],
       fill_value=999999)

# strange behavior with np.where

In [30]:
np.where(np.bitwise_and(m+1, 0x1))

(array([0, 2, 4, 7, 9]),)

In [31]:
np.where(np.bitwise_and(m+1, 0x1)==1)

(array([0, 2, 4]),)

In [32]:
np.where(np.array([0,1,-1,0,100]))

(array([1, 2, 4]),)

# test class method's variable scope

In [33]:
class s:
    def __init__(self):
        self.x = 10
        self.y = 5
        self.z = {1:10, 2:5}
        
    def func(self):
        my_list = [{1:10, 2:5}, {10:10, 5:5}]
        print(my_list)
        self.func2(my_list)
        print(my_list)
        self.func3(my_list[0])
        print(my_list)
    
    def func2(self, my_list):
        my_list[0][3] = 3.33
        print(my_list)
    
    def func3(self, my_dict):
        my_dict[3] = 1
        print(my_dict)

In [34]:
t = s()

In [35]:
t.func()

[{1: 10, 2: 5}, {10: 10, 5: 5}]
[{1: 10, 2: 5, 3: 3.33}, {10: 10, 5: 5}]
[{1: 10, 2: 5, 3: 3.33}, {10: 10, 5: 5}]
{1: 10, 2: 5, 3: 1}
[{1: 10, 2: 5, 3: 1}, {10: 10, 5: 5}]


# slicing

In [36]:
y = np.arange(10)

In [37]:
y

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [38]:
x[:0]

array([], dtype=int64)

In [39]:
y[0:]

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [40]:
np.concatenate((x[:0], y[0:]),axis=0)

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [41]:
x[np.where(y==10)] +=1

In [42]:
x[-1:]

array([9])

# split into integer and fraction

In [43]:
x = np.random.randint(low=0, high= 32000, size=150)
y = np.random.random(size=150)

idx_0 = np.random.randint(low=0, high= 149, size = 10)
idx_1 = np.random.randint(low=0, high= 149, size = 2)
y[idx_0] = 0
y[idx_1] = 1

z = x*2+y

In [44]:
z

array([49248.87656389, 37152.0069141 , 38270.67387788, 28750.11705099,
       58546.71484524, 16202.02183472, 53354.69068139,  3982.40835758,
       18762.13838737,  1566.81623389, 60086.3897546 ,  8196.53821044,
       49454.53706614, 17694.50664618, 23630.23960905, 22024.        ,
       39024.26810124, 28302.57866358, 23874.42400705, 61656.70075576,
       52966.59973846, 63484.43819831, 46490.58868491, 43956.85250957,
       39278.88462167, 17462.50465824, 23136.37412991, 41528.9324542 ,
        8395.        , 47902.30967369, 11176.09121129, 25626.15701634,
       59930.93562384,  2990.34256055, 26384.089998  , 38394.32321039,
       25154.        ,  5232.53755517, 21620.16885338, 33984.25780498,
       43714.        , 30774.17862759, 15118.54174676, 14480.65549332,
       43714.06295782, 57844.08677927,  4404.71181368, 22298.48185381,
        7502.62108991, 26358.50045141,  3298.39135227, 46458.15483793,
       60398.60120982, 25714.76655007, 47920.16860658, 19087.        ,
      

In [45]:
def split_intfrac(arr):
    part_int = np.bitwise_and(np.array(arr, dtype=np.int16), 0xfffe)
    part_frac= arr - part_int
    part_int = np.right_shift(part_int, 1)
    return part_int, part_frac

In [46]:
%timeit split_intfrac(z)

9.91 µs ± 16.6 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


In [47]:
%timeit np.modf(z)
## unfortunately, this can only handle (0,1) not [0,1]

3.1 µs ± 25.1 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


In [48]:
part_int, part_frac = split_intfrac(z)

In [49]:
print(np.all(part_int == x))
print(np.all(y - part_frac < 0.0000001)) # might have some very small rounding errors

True
True


# create numpy from string 

In [59]:
val_str = ','.join([str(i) for i in z[0:75]])
val_str2= val_str +','

In [66]:
%timeit np.fromstring(val_str, dtype=np.float32, sep=',')

25 µs ± 28.1 ns per loop (mean ± std. dev. of 7 runs, 10000 loops each)


In [60]:
%timeit np.fromstring(val_str, dtype=float, sep=',')

24.4 µs ± 48.9 ns per loop (mean ± std. dev. of 7 runs, 10000 loops each)


In [63]:
np.fromstring(val_str, dtype=float, sep=',')

array([49248.87656389, 37152.0069141 , 38270.67387788, 28750.11705099,
       58546.71484524, 16202.02183472, 53354.69068139,  3982.40835758,
       18762.13838737,  1566.81623389, 60086.3897546 ,  8196.53821044,
       49454.53706614, 17694.50664618, 23630.23960905, 22024.        ,
       39024.26810124, 28302.57866358, 23874.42400705, 61656.70075576,
       52966.59973846, 63484.43819831, 46490.58868491, 43956.85250957,
       39278.88462167, 17462.50465824, 23136.37412991, 41528.9324542 ,
        8395.        , 47902.30967369, 11176.09121129, 25626.15701634,
       59930.93562384,  2990.34256055, 26384.089998  , 38394.32321039,
       25154.        ,  5232.53755517, 21620.16885338, 33984.25780498,
       43714.        , 30774.17862759, 15118.54174676, 14480.65549332,
       43714.06295782, 57844.08677927,  4404.71181368, 22298.48185381,
        7502.62108991, 26358.50045141,  3298.39135227, 46458.15483793,
       60398.60120982, 25714.76655007, 47920.16860658, 19087.        ,
      

In [61]:
%timeit np.fromstring(val_str2,  dtype=float, sep=',')

24.9 µs ± 74.4 ns per loop (mean ± std. dev. of 7 runs, 10000 loops each)


In [62]:
np.fromstring(val_str2,  dtype=float, sep=',')

array([49248.87656389, 37152.0069141 , 38270.67387788, 28750.11705099,
       58546.71484524, 16202.02183472, 53354.69068139,  3982.40835758,
       18762.13838737,  1566.81623389, 60086.3897546 ,  8196.53821044,
       49454.53706614, 17694.50664618, 23630.23960905, 22024.        ,
       39024.26810124, 28302.57866358, 23874.42400705, 61656.70075576,
       52966.59973846, 63484.43819831, 46490.58868491, 43956.85250957,
       39278.88462167, 17462.50465824, 23136.37412991, 41528.9324542 ,
        8395.        , 47902.30967369, 11176.09121129, 25626.15701634,
       59930.93562384,  2990.34256055, 26384.089998  , 38394.32321039,
       25154.        ,  5232.53755517, 21620.16885338, 33984.25780498,
       43714.        , 30774.17862759, 15118.54174676, 14480.65549332,
       43714.06295782, 57844.08677927,  4404.71181368, 22298.48185381,
        7502.62108991, 26358.50045141,  3298.39135227, 46458.15483793,
       60398.60120982, 25714.76655007, 47920.16860658, 19087.        ,
      